# 5일차 실습 1. LLM 호출 로직 모듈화

## 실습 목표

이번 실습은 4일차의 `/chat/stream/robust` 한 함수에 다 들어 있던 코드를 **Router / Service / Client 세 층으로 분리**하는 단계입니다. 환경 변수와 Pydantic 모델도 각자의 자리(`core/`, `schemas/`)로 옮겨, 한 파일에 몰려 있던 코드가 폴더 구조 위에서 어떻게 흩어지는지 직접 익힙니다. 이번 실습이 끝나면 같은 구조 위에 시스템 프롬프트·비스트리밍 호출·새 엔드포인트를 자연스럽게 얹을 수 있는 기반이 마련됩니다.

진행 흐름은 다음과 같습니다.

- `/chat/stream/robust` 한 함수의 코드를 Router / Service / Client 세 층으로 분리
- 환경 변수는 `core/config.py`로, Pydantic 모델은 `schemas/`로 자리 옮기기
- 분리 후에도 동작이 동일함을 호출 셀로 검증 (정상 호출 / 잘못된 model / 클라이언트 중단)
- TODO를 통해 시스템 프롬프트 적용, 비스트리밍 호출 함수 추가, 비스트리밍 엔드포인트 추가까지 손에 익힘

## 1. 환경 준비

FastAPI, uvicorn, httpx, openai, pydantic, dotenv를 사용합니다.

In [19]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

base_url   = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1")
api_key    = os.getenv("MLAPI_API_KEY")
model_name = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not api_key or api_key.startswith("여기에"):
    api_key = getpass("MLAPI API Key를 입력하세요: ")

# config.py가 읽을 수 있도록 .env 파일에 저장
Path(".env").write_text(
    f"MLAPI_BASE_URL={base_url}\n"
    f"MLAPI_API_KEY={api_key}\n"
    f"MLAPI_MODEL={model_name}\n",
    encoding="utf-8",
)
Path(".gitignore").write_text(".env\n__pycache__/\n", encoding="utf-8")

PORT = 8000                  # 서버 포트. 충돌나면 8001 등으로 변경
SERVER_BASE = f"http://localhost:{PORT}"

print("모델명:", model_name, "/ PORT:", PORT)
print("server base:", SERVER_BASE)
print(".env 저장 완료 — config.py가 이 파일에서 키를 읽습니다")

모델명: openai/gpt-4o-mini / PORT: 8000
server base: http://localhost:8000
.env 저장 완료 — config.py가 이 파일에서 키를 읽습니다


## 2. 프로젝트 폴더 / 빈 파일 자동 생성

이번 실습부터는 한 파일(`server.py`) 이 아니라 폴더 구조로 갑니다. 아래 셀이 `app/` 트리와 빈 `__init__.py` 들을 한 번에 만들어 줍니다.

```
app/
  __init__.py
  main.py
  core/
    __init__.py
    config.py
  schemas/
    __init__.py
    chat.py
  clients/
    __init__.py
    openai_client.py
  services/
    __init__.py
    chat_service.py
  routers/
    __init__.py
    chat.py
```

In [7]:
from pathlib import Path

ROOT = Path("app")
SUBDIRS = ["core", "schemas", "clients", "services", "routers"]

ROOT.mkdir(exist_ok=True)
(ROOT / "__init__.py").touch()
for name in SUBDIRS:
    (ROOT / name).mkdir(exist_ok=True)
    (ROOT / name / "__init__.py").touch()

print("폴더 / __init__.py 생성 완료")
for p in sorted(ROOT.rglob("*")):
    print("  -", p)

폴더 / __init__.py 생성 완료
  - app/__init__.py
  - app/clients
  - app/clients/__init__.py
  - app/core
  - app/core/__init__.py
  - app/routers
  - app/routers/__init__.py
  - app/schemas
  - app/schemas/__init__.py
  - app/services
  - app/services/__init__.py


## 3. app/core/config.py — 환경 변수 한 곳에 모음

`API_KEY`, `MODEL`, `BASE_URL` 같은 설정 값을 한 모듈에서 정의합니다. 다른 모듈은 여기서 import 만 해서 씁니다. 값이 바뀔 때 고쳐야 할 곳이 한 군데로 줄어듭니다.

In [8]:
from pathlib import Path

code = '''import os
from dotenv import load_dotenv

load_dotenv()

API_KEY  = os.getenv("MLAPI_API_KEY")
MODEL    = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")
BASE_URL = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1")

if not API_KEY:
    raise RuntimeError("MLAPI_API_KEY가 설정되어 있지 않습니다. .env 파일을 확인하세요.")
'''

Path("app/core/config.py").write_text(code, encoding="utf-8")
print("app/core/config.py 작성 완료")

app/core/config.py 작성 완료


## 4. app/schemas/chat.py — 요청 모델

`ChatRequest` Pydantic 모델만 분리해 둡니다. Router 와 Service 가 이 모델을 공유해서 씁니다. 응답 모델은 스트리밍이라 별도로 두지 않습니다.

In [9]:
from pathlib import Path

code = '''
from typing import Optional
from pydantic import BaseModel, Field

class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1)
    model: Optional[str] = Field(None)
    temperature: Optional[float] = Field(None, ge=0.0, le=2.0)

'''

Path("app/schemas/chat.py").write_text(code, encoding="utf-8")
print("app/schemas/chat.py 작성 완료")

app/schemas/chat.py 작성 완료


## 5. app/clients/openai_client.py — LLM 호출 한 곳에

`AsyncOpenAI` 인스턴스 생성과 스트리밍 호출을 이 모듈에 모읍니다. Service 가 OpenAI SDK 를 직접 import 하지 않도록 하기 위함입니다. 모델 공급자 / SDK 가 바뀌면 여기만 바뀌면 됩니다.

- `get_client()` : `AsyncOpenAI` 인스턴스 반환
- `stream_chat(...)` : 메시지 리스트를 받아 토큰을 async generator 로 yield

In [10]:
from pathlib import Path

code = '''
from typing import AsyncIterator, List, Dict, Optional
from openai import AsyncOpenAI
from app.core.config import API_KEY, MODEL, BASE_URL

def get_client() -> AsyncOpenAI:
    if BASE_URL:
        return AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)
    return AsyncOpenAI(api_key=API_KEY)

_client = get_client()

async def stream_chat(
    messages: List[Dict[str, str]],
    model: Optional[str] = None,
    temperature: Optional[float] = None,
    timeout: float = 10.0,
) -> AsyncIterator[str]:
    """OpenAI 스트리밍 호출. 토큰 piece 만 yield 한다."""
    stream = await _client.chat.completions.create(
        model=model or MODEL,
        messages=messages,
        temperature=temperature if temperature is not None else 1.0,
        stream=True,
        timeout=timeout,
    )
    async for chunk in stream:
        if not chunk.choices:
            continue
        piece = getattr(chunk.choices[0].delta, "content", None)
        if not piece:
            continue
        yield piece

'''

Path("app/clients/openai_client.py").write_text(code, encoding="utf-8")
print("app/clients/openai_client.py 작성 완료")

app/clients/openai_client.py 작성 완료


## 6. app/services/chat_service.py — 비즈니스 로직

Service 는 HTTP 를 모릅니다. `ChatRequest` 객체를 받아 메시지 리스트로 변환하고, Client 의 `stream_chat(...)` 을 호출해 토큰을 받아옵니다.

안전 패턴은 여기로 옮깁니다.

- `produced` 플래그로 빈 응답 감지
- 정상 종료 → `[DONE]`, 빈 응답 → `[EMPTY]`, 예외 → `[ERROR] ...`
- `asyncio.CancelledError` 잡아서 로그 남기고 `raise`
- `finally` 에서 정리 로그

In [11]:
from pathlib import Path

code = '''
import asyncio
from typing import AsyncIterator
from app.schemas.chat import ChatRequest
from app.clients.openai_client import stream_chat

def _build_messages(req: ChatRequest):
    return [{"role": "user", "content": req.message}]



async def stream_chat_safe(req: ChatRequest) -> AsyncIterator[str]:
    produced = False
    try:
        async for piece in stream_chat(
            messages=_build_messages(req),
            model=req.model,
            temperature=req.temperature,
        ):
            produced = True
            yield piece
        if produced:
            yield "\\n[DONE]"
        else:
            yield "[EMPTY]"
            yield "\\n"
    except asyncio.CancelledError:
        print("[chat_service] cancelled by client")
        raise
    except Exception as e:
        yield f"[ERROR] {type(e).__name__}: {e}"
        yield "\\n"
    finally:
        print("[chat_service] generator finished or cancelled")


'''

Path("app/services/chat_service.py").write_text(code, encoding="utf-8")
print("app/services/chat_service.py 작성 완료")

app/services/chat_service.py 작성 완료


## 7. app/routers/chat.py — HTTP 입출력만

Router 는 HTTP 만 책임집니다. `ChatRequest` 로 요청을 받고, Service 의 async generator 를 `StreamingResponse` 에 넘기기만 합니다. 비즈니스 로직은 없습니다.

In [12]:
from pathlib import Path

code = '''
from fastapi import APIRouter
from fastapi.responses import StreamingResponse
from app.schemas.chat import ChatRequest
from app.services.chat_service import stream_chat_safe

router = APIRouter()

@router.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    return StreamingResponse(stream_chat_safe(req), media_type="text/plain")

'''

Path("app/routers/chat.py").write_text(code, encoding="utf-8")
print("app/routers/chat.py 작성 완료")

app/routers/chat.py 작성 완료


## 8. app/main.py — 앱 조립

`FastAPI()` 앱을 만들고, Router 를 `include_router(...)` 로 등록합니다. main 에는 비즈니스 로직이 들어가지 않습니다.

In [13]:
from pathlib import Path

code = '''
from fastapi import FastAPI
from app.routers.chat import router as chat_router

app = FastAPI()
app.include_router(chat_router)

@app.get("/health")
def health():
    return {"ok": True}

'''

Path("app/main.py").write_text(code, encoding="utf-8")
print("app/main.py 작성 완료")

app/main.py 작성 완료


## 9. 서버 띄우기

터미널에서 아래 셀이 출력하는 명령어를 그대로 붙여넣어 실행하세요.

- 이번엔 `server.py` 가 아니라 `app.main:app` 경로입니다.
- `--reload` 로 어떤 파일을 고쳐도 자동 재시작됩니다.
- 포트 충돌 시 1 단계 `PORT` 만 바꾸고 3, 9 단계 셀 재실행.

In [20]:
print(f"uvicorn app.main:app --reload --port {PORT}")
print()
print("서버 주소:", SERVER_BASE)

uvicorn app.main:app --reload --port 8000

서버 주소: http://localhost:8000


In [24]:
import httpx

r = httpx.get(f"{SERVER_BASE}/health", timeout=5.0)
print(r.status_code, r.json())

200 {'ok': True}


## 10. 정상 호출 검증

Router → Service → Client 가 정상 동작하면, 본문 끝이 `[DONE]` 으로 끝나야 합니다.

In [25]:
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream",
        json={"message": "FastAPI 한 줄 요약 부탁해"},
    ) as r:
        print("status:", r.status_code)
        body = ""
        for piece in r.iter_text():
            body += piece
            print(piece, end="", flush=True)
print("\n--- end ---")
assert body.rstrip().endswith("[DONE]"), "정상 응답은 [DONE] 으로 끝나야 합니다."

status: 200
FastAPI는 Python으로 구축된 웹 프레임워크로, 빠른 API 개발을 지원하며 비동기 프로그래밍과 데이터 검증 기능을 제공합니다.
[DONE]
--- end ---


## 11. 잘못된 model 호출 검증

Service 의 `except Exception` 에 걸려 본문 끝에 `[ERROR] ...` 가 흘러야 합니다. 서버는 죽지 않습니다.

In [26]:
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream",
        json={"message": "테스트", "model": "this-model-does-not-exist"},
    ) as r:
        print("status:", r.status_code)
        body = ""
        for piece in r.iter_text():
            body += piece
            print(piece, end="", flush=True)
print("\n--- end ---")
assert "[ERROR]" in body, "호출 실패 시 본문에 [ERROR] 마커가 있어야 합니다."

status: 200
[ERROR] BadRequestError: Error code: 400 - {'detail': "Model 'this-model-does-not-exist' is not allowed. Allowed models: ['openai/gpt-4o-mini']"}

--- end ---


## 12. 클라이언트 중단 검증

클라이언트가 도중에 `break` 로 응답 수신을 끊으면, 서버의 Service 가 `CancelledError` 를 받아 로그를 남기고 정리합니다.

서버 터미널 로그에 다음 두 줄이 차례로 찍히는지 확인하세요.

- `[chat_service] cancelled by client`
- `[chat_service] generator finished or cancelled`

In [27]:
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream",
        json={"message": "한국의 역사를 가능한 한 길게 설명해줘"},
    ) as r:
        count = 0
        for piece in r.iter_text():
            print(piece, end="", flush=True)
            count += 1
            if count >= 5:
                print("\n[client] break")
                break
print("--- end ---")
print("서버 터미널에서 [chat_service] cancelled by client → [chat_service] generator finished or cancelled 순으로 로그를 확인하세요.")

한국의 역사는 매우
[client] break
--- end ---
서버 터미널에서 [chat_service] cancelled by client → [chat_service] generator finished or cancelled 순으로 로그를 확인하세요.


# TODO

지금까지 본 패턴을 직접 손에 익히기 위한 과제입니다.

## TODO 1. Service 에 시스템 프롬프트 적용

지금 `_build_messages` 는 `messages` 에 `user` 메시지 한 줄만 담습니다.
모델의 역할·말투를 고정하려면 그 앞에 `system` 메시지가 필요합니다.

아래 셀에서 두 곳을 채우고 실행하세요.

- **TODO 1-1** — `SYSTEM_PROMPT`: 모델의 역할·말투를 정하는 지시 문자열 (내용 자유, 예: 친절한 한국어 백엔드 튜터)
- **TODO 1-2** — `messages` 리스트의 첫 원소: `system` 역할 메시지. 바로 아래 `user` 원소가 형태 참고가 됩니다.

`stream_chat_safe` 등 나머지 코드는 건드리지 않습니다.

In [28]:
# TODO 1
todo1_code = '''
# TODO 1-1: 모델의 역할·말투를 정하는 시스템 프롬프트 문자열
SYSTEM_PROMPT = None


def _build_messages(req: ChatRequest):
    return [
        # TODO 1-2: 첫 원소로 system 역할 메시지 (아래 user 원소가 형태 참고)
        None,
        {"role": "user", "content": req.message},
    ]'''

# (아래는 그대로 실행)
from pathlib import Path
path = Path("app/services/chat_service.py")
base = 'def _build_messages(req: ChatRequest):\n    return [{"role": "user", "content": req.message}]'
path.write_text(path.read_text(encoding="utf-8").replace(base, todo1_code), encoding="utf-8")
print("적용 완료")

적용 완료


In [30]:
# TODO 1 확인 — _build_messages 가 system + user 2개를 만드는지
import importlib
import app.services.chat_service as chat_service
importlib.reload(chat_service)   # 방금 수정한 _build_messages 반영
from app.schemas.chat import ChatRequest

msgs = chat_service._build_messages(ChatRequest(message="테스트"))
for m in msgs:
    print(m)

assert len(msgs) == 2, "system 1 + user 1, 총 2개여야 합니다"
assert isinstance(msgs[0], dict) and msgs[0].get("role") == "system", "첫 메시지는 role 이 system 인 dict 여야 합니다 (TODO 1-2)"
assert isinstance(msgs[0].get("content"), str) and msgs[0]["content"], "system 메시지 content 는 비어있지 않은 문자열이어야 합니다 (TODO 1-1)"
assert msgs[1].get("role") == "user", "둘째 메시지는 role 이 user 여야 합니다"
print("TODO 1 OK")

{'role': 'system', 'content': '당신은 친절한 조수입니다. 적확하게 설명하지만 초보가 이해할 수 있도록 간단하고 명료하게 답하는 성격입니다.'}
{'role': 'user', 'content': '테스트'}
TODO 1 OK


## TODO 2. Client 에 비스트리밍 호출 함수 추가

`stream_chat` 은 토큰을 흘려보냈습니다. 이번엔 답변을 한 번에 받아 문자열로 돌려주는 `complete_once` 를 추가합니다.

아래 셀에서 `return` 한 곳을 채우고 실행하세요.

- **TODO 2** — `resp` 에서 답변 문자열을 꺼내 반환합니다. 스트리밍에서는 `chunk.choices[0].delta.content` 를 모았지만, 단발 응답 `resp` 에는 `delta` 가 없습니다 — 완성된 답변이 어디 들어있는지 찾아보세요.

`stream=False` 등 나머지는 `stream_chat` 과 같은 패턴이라 채워 두었습니다.

In [31]:
# TODO 2
todo2_code = '''
async def complete_once(
    messages: List[Dict[str, str]],
    model: Optional[str] = None,
    temperature: Optional[float] = None,
    timeout: float = 10.0,
) -> str:
    """비스트리밍 단발 호출. 최종 답변 문자열만 반환한다."""
    resp = await _client.chat.completions.create(
        model=model or MODEL,
        messages=messages,
        temperature=temperature if temperature is not None else 1.0,
        stream=False,   # 단발 호출 — stream_chat 의 stream=True 와 반대
        timeout=timeout,
    )

    # TODO 2: resp 에서 답변 문자열을 꺼내 반환 (스트리밍은 delta.content, 단발은?)
    return None
    
'''

with open("app/clients/openai_client.py", "a", encoding="utf-8") as f:
    f.write(todo2_code)
print("complete_once 추가 완료")

complete_once 추가 완료


In [32]:
# TODO 2 확인 — complete_once 가 답변 문자열을 반환하는지
import importlib
import app.clients.openai_client as openai_client
importlib.reload(openai_client)   # 방금 추가한 complete_once 를 노트북에 반영

reply = await openai_client.complete_once(
    messages=[
        {"role": "system", "content": "너는 친절한 한국어 백엔드 튜터야."},
        {"role": "user", "content": "FastAPI 한 줄 요약 부탁해"},
    ],
)
print("reply:", reply)
assert isinstance(reply, str) and reply, "complete_once 가 비어있지 않은 문자열을 반환해야 합니다"

reply: FastAPI는 Python으로 비동기 웹 API를 쉽게 작성할 수 있게 해주는 고속 웹 프레임워크입니다.


## TODO 3. Router 에 비스트리밍 엔드포인트 추가

지금까지 만든 층을 묶어 `/chat` (비스트리밍, JSON 응답) 엔드포인트를 만듭니다.
Router → Service → Client 흐름을 한 번 더 직접 짭니다.

아래 두 셀의 빈칸을 채우고 실행하세요.

- **TODO 3-1** (`chat_service.py`) — `chat_once(req)`: `_build_messages` 로 메시지를 만들고 `complete_once` 로 호출해 답변 문자열을 반환합니다.
- **TODO 3-2** (`routers/chat.py`) — `POST /chat`: `chat_once` 를 호출해 `{"reply": "..."}` 형태로 반환합니다.

기존 `/chat/stream` 은 그대로 둡니다.

In [33]:
# TODO 3-1  (답안지)
todo3_1_code = '''
from app.clients.openai_client import complete_once

async def chat_once(req: ChatRequest) -> str:
    # TODO 3-1: complete_once 를 호출해 답변 문자열을 반환
    #   messages는 _build_messages(req), model·temperature 는 req 에서
    #   (위 stream_chat_safe 의 stream_chat 호출이 형태 참고)
    return None(
        messages=None,
        model=None,
        temperature=None,
    )
'''

with open("app/services/chat_service.py", "a", encoding="utf-8") as f:
    f.write(todo3_1_code)
print("chat_service.py 에 chat_once 추가 완료")

chat_service.py 에 chat_once 추가 완료


In [34]:
# TODO 3-2  (학생용)
chat_router_code = '''from fastapi import APIRouter
from fastapi.responses import StreamingResponse
from app.schemas.chat import ChatRequest
from app.services.chat_service import stream_chat_safe, chat_once

router = APIRouter()


@router.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    return StreamingResponse(stream_chat_safe(req), media_type="text/plain")


@router.post("/chat")
async def chat(req: ChatRequest):
    # TODO 3-2: chat_once 를 호출하고 결과를 {"reply": ...} 형태 dict 로 반환
    return None
'''

with open("app/routers/chat.py", "w", encoding="utf-8") as f:
    f.write(chat_router_code)
print("routers/chat.py 갱신 완료 — /chat 추가됨")

routers/chat.py 갱신 완료 — /chat 추가됨


In [35]:
# TODO 3 확인 — /chat 가 {"reply": "..."} 를 반환하는지
import httpx

r = httpx.post(
    f"{SERVER_BASE}/chat",
    json={"message": "FastAPI 한 줄 요약 부탁해"},
    timeout=30.0,
)
print("status:", r.status_code)
assert r.status_code == 200, f"/chat 가 200 을 반환해야 합니다 (현재 {r.status_code})"

data = r.json()
print("reply:", data)
assert isinstance(data, dict) and isinstance(data.get("reply"), str) and data["reply"], \
    '응답은 {"reply": "답변 문자열"} 형태여야 합니다'
print("TODO 3 OK")

status: 200
reply: {'reply': 'FastAPI는 빠르고 효율적인 웹 API를 쉽게 개발할 수 있게 도와주는 Python 기반의 프레임워크입니다.'}
TODO 3 OK


## 정리

- 한 함수에 다 있던 코드를 Router / Service / Client / Schema / Config 다섯 자리로 나눠 본 경험.
- 의존성 흐름이 한 방향(`Router → Service → Client`) 으로 유지된 것을 import 위치로 확인할 수 있다.
- 프롬프트 변경은 Service 한 곳만, LLM 호출 방식 변경은 Client 한 곳만, HTTP 모양 변경은 Router 한 곳만 — 바꿔야 할 자리가 명확해진다.
- 같은 Service 층을 스트리밍(`/chat/stream`)·비스트리밍(`/chat`) 두 엔드포인트가 함께 사용한다 — 모듈화의 실용적 이득.
- 다음 단계(프롬프트 관리·로깅·캐싱·미니 프로젝트) 를 얹을 자리가 이미 마련되어 있다.